# Alia on a Colab T4: 35B Q4 with 16 GB VRAM

Measures Qwen3.6-35B-A3B **UD-Q4_K_XL** (22.4 GB) on a free Colab T4 (16 GB VRAM) with the llama.cpp build Alia pins
(b11639 = a0e6dc7), at the context of the `rtx12-16gb-qwen36-35b-q4` profile (32,768).

**Variant, not the exact profile.** The profile needs 30 GB RAM; Colab free has 12.7 GB. With the profile's exact
`server_args` the server is OOM-killed while loading (exit -9). This notebook adds `--no-repack --no-host`, so the
expert weights that do not fit in VRAM stay memory-mapped from disk instead of being copied into RAM. Numbers from
here are labelled `colab-t4-12-16gb-norepack`.

**How to run**
1. *Runtime → Change runtime type → T4 GPU*.
2. *Runtime → Run all*. Download plus measurement take roughly 20–30 minutes.
3. Copy the output of the last cell.

## 1. Machine

In [ ]:
import subprocess, shutil
assert shutil.which("nvidia-smi"), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())
!free -g | head -2
!df -h /content | tail -1
!nproc

## 2. Setup: llama.cpp b11639, model, Alia (skips what is already there)

In [ ]:
%%bash
set -e
cd /content
if [ ! -d llama ]; then
  R=https://github.com/ggml-org/llama.cpp/releases/download/b11639
  wget -q $R/llama-b11639-bin-ubuntu-cuda-12.8-x64.tar.gz $R/cudart-llama-b11639-bin-ubuntu-cuda-12.8-x64.tar.gz
  mkdir llama
  tar xzf llama-b11639-bin-ubuntu-cuda-12.8-x64.tar.gz -C llama
  tar xzf cudart-llama-b11639-bin-ubuntu-cuda-12.8-x64.tar.gz -C llama
fi
BIN=$(find /content/llama -name llama-server -type f | head -1)
LD_LIBRARY_PATH=$(dirname $BIN) $BIN --version 2>&1 | tail -2

pip -q install -U huggingface_hub
HF_XET_HIGH_PERFORMANCE=1 hf download unsloth/Qwen3.6-35B-A3B-GGUF Qwen3.6-35B-A3B-UD-Q4_K_XL.gguf \
  --local-dir /content/models > /dev/null
ls -l /content/models/*.gguf

[ -d alia ] || git clone -q --branch micro/r10 https://github.com/BEKO2210/alia
cd alia && git pull -q && git log --oneline -1

## 3. Measure (3 reps at 4K, 16K and 28K tokens of context)

In [ ]:
%%bash
cd /content/alia
export LLAMA_SERVER=$(find /content/llama -name llama-server -type f | head -1)
free -g | head -2
# profile server_args of rtx12-16gb-qwen36-35b-q4 + --no-repack --no-host (see top); no -ngl, llama.cpp --fit decides
python3 bench/bench_llamacpp.py --gguf /content/models/Qwen3.6-35B-A3B-UD-Q4_K_XL.gguf \
  --name colab-t4-12-16gb-norepack --configs fit --depths 4096,16384,28000 --reps 3 --ctx 32768 \
  --extra "-b 2048 -ub 2048 --jinja --cache-ram 2048 --no-repack --no-host" 2>&1 | tail -8

## 4. Result: copy everything below

In [ ]:
import json, glob, os, re
os.chdir("/content/alia")
rows = []
for f in sorted(glob.glob("bench/results/llamacpp-*.jsonl"), key=os.path.getmtime):
    rows += [r for r in map(json.loads, open(f)) if r.get("model") == "colab-t4-12-16gb-norepack"]
if not rows:
    print("No result rows. Server log tail:")
    print("".join(open("bench/results/llama-server-colab-t4-12-16gb-norepack-fit.log").readlines()[-25:]))
else:
    print(f"{'depth':>6} {'pp tok/s':>9} {'tg tok/s':>9} {'VRAM MiB':>9} {'RSS MiB':>8} {'load s':>7}")
    for r in rows:
        if "pp_tps" in r:
            print(f"{r['depth']:>6} {r['pp_tps']['median']:>9.0f} {r['tg_tps']['median']:>9.1f} "
                  f"{r['vram_peak_mib']:>9.0f} {r.get('rss_peak_mib', 0):>8} {r['load_s']:>7.0f}")
        else:
            print("ERROR", r.get("error", r))
    log = open("bench/results/llama-server-colab-t4-12-16gb-norepack-fit.log").read()
    print("\n--- offload / buffers")
    print("\n".join(l for l in log.splitlines() if re.search(r"offload|model buffer size|KV buffer size|fit", l))[-2500:])
    print("\n--- raw JSONL")
    for r in rows:
        print(json.dumps(r))